# WILDTRACK Dataset Download & Sample
Downloads WILDTRACK (EPFL CVLAB) dataset, generates MP4 videos from frame images,
and packages a sample for local testing.

**Output**: `wildtrack_sample.zip` with:
- All 7 cameras as MP4 video
- Ground truth annotation JSON files

In [ ]:
import os, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
TMP = Path("/tmp/datasets")
TMP.mkdir(parents=True, exist_ok=True)

for m in ["/kaggle/working", "/tmp"]:
    t, u, f = shutil.disk_usage(m)
    print(f"{m}: {f/(1024**3):.1f} GB free")

In [ ]:
!pip install -q opencv-python-headless 2>&1 | tail -3
!apt-get install -y -qq p7zip-full 2>&1 | tail -3

## 1. Download WILDTRACK (~6.3 GB)

In [ ]:
import urllib.request

WT_URL = "https://documents.epfl.ch/groups/c/cv/cvlab-unit/www/data/Wildtrack/Wildtrack_dataset_full.zip"
WT_DIR = TMP / "wildtrack"
WT_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE = WT_DIR / "Wildtrack_dataset_full.zip"

if ARCHIVE.exists():
    print(f"Archive already exists: {ARCHIVE} ({ARCHIVE.stat().st_size/(1024**3):.2f} GB)")
else:
    print(f"Downloading WILDTRACK from {WT_URL}...")
    print("This will take a while (~6.3 GB)...")
    
    _last_pct = [-1]
    def _progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            pct = min(100, downloaded * 100 // total_size)
            if pct > _last_pct[0]:
                _last_pct[0] = pct
                mb = downloaded / (1024**2)
                total_mb = total_size / (1024**2)
                print(f"  {mb:.0f} / {total_mb:.0f} MB ({pct}%)")
    
    urllib.request.urlretrieve(WT_URL, str(ARCHIVE), reporthook=_progress)
    print(f"\nDownloaded: {ARCHIVE.stat().st_size/(1024**3):.2f} GB")

## 2. Extract (using 7z — handles ZIP64 archives >4 GB)

In [ ]:
import subprocess

# Check if already extracted
img_subsets = WT_DIR / "Image_subsets"
if img_subsets.exists():
    print("Already extracted, skipping.")
else:
    # Use 7z instead of unzip/Python zipfile — both choke on ZIP64 (>4 GB)
    # unzip: "invalid zip file with overlapped components"
    # Python zipfile: "BadZipFile: Bad magic number for file header"
    print("Extracting with 7z (ZIP64-safe)...")
    result = subprocess.run(
        ["7z", "x", "-y", f"-o{WT_DIR}", str(ARCHIVE)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"7z STDERR:\n{result.stderr[-2000:]}")
        raise RuntimeError(f"7z failed with code {result.returncode}")
    
    # The zip may extract into a subfolder — find and hoist if needed
    if not img_subsets.exists():
        for child in WT_DIR.iterdir():
            if child.is_dir() and (child / "Image_subsets").exists():
                print(f"  Moving contents from {child.name}/ up...")
                for item in child.iterdir():
                    dest = WT_DIR / item.name
                    if not dest.exists():
                        shutil.move(str(item), str(dest))
                shutil.rmtree(child, ignore_errors=True)
                break
    
    print("Extraction complete.")

In [ ]:
# Verify structure
print("WILDTRACK contents:")
for d in ["Image_subsets", "annotations_positions"]:
    p = WT_DIR / d
    if p.exists():
        items = sorted(p.iterdir())
        print(f"  {d}: {len(items)} items")
        for item in items[:10]:
            if item.is_dir():
                n = len(list(item.glob("*.png")))
                print(f"    {item.name}: {n} frames")
            else:
                print(f"    {item.name}")
        if len(items) > 10:
            print(f"    ... and {len(items)-10} more")
    else:
        print(f"  {d}: NOT FOUND")

## 3. Generate MP4 videos from frame sequences
WILDTRACK provides PNG frames (sampled at 2fps from a 60fps stream).
We encode them as MP4 so our pipeline's video-based ingestion can use them.

In [ ]:
import cv2

img_root = WT_DIR / "Image_subsets"
videos_dir = WT_DIR / "videos"
videos_dir.mkdir(exist_ok=True)

cams = sorted(d for d in img_root.iterdir() if d.is_dir())
print(f"Found {len(cams)} cameras")

for cam_dir in cams:
    frames = sorted(cam_dir.glob("*.png"))
    if not frames:
        continue
    
    video_path = videos_dir / f"{cam_dir.name}.mp4"
    if video_path.exists():
        print(f"  {video_path.name} already exists, skipping")
        continue
    
    sample = cv2.imread(str(frames[0]))
    if sample is None:
        print(f"  WARNING: Could not read {frames[0]}")
        continue
    h, w = sample.shape[:2]
    
    # WILDTRACK is 2 fps
    writer = cv2.VideoWriter(
        str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), 2.0, (w, h)
    )
    for fp in frames:
        img = cv2.imread(str(fp))
        if img is not None:
            writer.write(img)
    writer.release()
    
    size_mb = video_path.stat().st_size / (1024**2)
    print(f"  {video_path.name}: {len(frames)} frames, {w}x{h}, {size_mb:.0f} MB")

In [ ]:
# Clean up archive to save space
if ARCHIVE.exists():
    print(f"Removing archive ({ARCHIVE.stat().st_size/(1024**3):.2f} GB)...")
    ARCHIVE.unlink()
    print("Done.")

## 4. Build Sample Package
Package all 7 cameras (MP4 videos + annotations) for download.

In [ ]:
import zipfile

SAMPLE_DIR = Path("/tmp/wt_sample")
if SAMPLE_DIR.exists():
    shutil.rmtree(SAMPLE_DIR)

wt_sample = SAMPLE_DIR / "wildtrack"

# Copy videos
vid_dir = WT_DIR / "videos"
if vid_dir.exists():
    dest_vid = wt_sample / "videos"
    dest_vid.mkdir(parents=True, exist_ok=True)
    for v in sorted(vid_dir.glob("*.mp4")):
        shutil.copy2(v, dest_vid / v.name)
        print(f"  {v.name}: {v.stat().st_size/(1024**2):.0f} MB")

# Copy annotations
ann_dir = WT_DIR / "annotations_positions"
if ann_dir.exists():
    dest_ann = wt_sample / "annotations_positions"
    shutil.copytree(ann_dir, dest_ann)
    n = len(list(dest_ann.glob("*.json")))
    print(f"  annotations: {n} JSON files")

total = sum(f.stat().st_size for f in SAMPLE_DIR.rglob("*") if f.is_file())
print(f"\nTotal sample size: {total/(1024**3):.2f} GB")

In [ ]:
ZIP_PATH = WORK / "wildtrack_sample.zip"

print("Creating zip...")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_STORED) as zf:
    for f in sorted(SAMPLE_DIR.rglob("*")):
        if f.is_file():
            arcname = str(f.relative_to(SAMPLE_DIR))
            zf.write(f, arcname)

print(f"\nDone: {ZIP_PATH.name} ({ZIP_PATH.stat().st_size/(1024**3):.2f} GB)")
print("Download from the Output tab on the right.")
print("\nExtract to data/raw/ in your local project:")
print("  data/raw/wildtrack/videos/C1.mp4 ...")
print("  data/raw/wildtrack/annotations_positions/*.json")

In [ ]:
# Summary
print("=" * 60)
print("WILDTRACK SAMPLE CONTENTS")
print("=" * 60)

for item in sorted(SAMPLE_DIR.rglob("*")):
    if item.is_file():
        rel = item.relative_to(SAMPLE_DIR)
        size = item.stat().st_size / (1024**2)
        if size > 0.1:
            print(f"  {rel} ({size:.1f} MB)")

json_count = len(list(SAMPLE_DIR.rglob("*.json")))
if json_count > 10:
    print(f"  ... + {json_count} JSON annotation files")